## Configuration

In [1]:
import os
from getpass import getpass

# Loading variables
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# API Keys
REQUIRED_KEYS = {
    "GROQ_API_KEY": "Groq API Key",
    "TAVILY_API_KEY": "Tavily API Key",
    "PINECONE_API_KEY": "Pinecone API Key"
}


for env_var, label in REQUIRED_KEYS.items():
    if not os.getenv(env_var):
        os.environ[env_var] = getpass(f"Enter {label}: ")

# Console Output
print("\n" + "═" * 55)
print("  SYSTEM CONFIGURATION & API STATUS")
print("═" * 55)

for env_var, label in REQUIRED_KEYS.items():
    is_active = bool(os.getenv(env_var))
    status_icon = "✔ READY" if is_active else "✖ MISSING"
    print(f"  • {label:<22} : [{status_icon}]")

print("═" * 55 + "\n")


═══════════════════════════════════════════════════════
  SYSTEM CONFIGURATION & API STATUS
═══════════════════════════════════════════════════════
  • Groq API Key           : [✔ READY]
  • Tavily API Key         : [✔ READY]
  • Pinecone API Key       : [✔ READY]
═══════════════════════════════════════════════════════



## Load Documents

In [1]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_paths=(SOURCE_URL,),
    requests_kwargs={
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    },
)

raw_docs = loader.load()

# Clean up leading/trailing whitespace from the raw content preview
preview_content = raw_docs[0].page_content[:1500].strip()

# Console output
print("=" * 60)
print(" 📄 DOCUMENT LOADING REPORT")
print("=" * 60)
print(f" ► Total Loaded Documents : {len(raw_docs)}")
print(f" ► Source URL             : {raw_docs[0].metadata.get('source')}")
print("-" * 60)
print(" 📝 CONTENT PREVIEW (First 1500 Characters)")
print("-" * 60)
print(preview_content)
print("=" * 60)

C:\Users\musta\AppData\Local\Temp\ipykernel_1944\405625008.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\musta\OneDrive\Desktop\Empower AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


 📄 DOCUMENT LOADING REPORT
 ► Total Loaded Documents : 1
 ► Source URL             : https://docs.langchain.com/oss/python/langgraph/agentic-rag
------------------------------------------------------------
 📝 CONTENT PREVIEW (First 1500 Characters)
------------------------------------------------------------
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom

## Chunking

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = splitter.split_documents(raw_docs)

# Clean up leading/trailing whitespace from the first chunk preview
chunk_preview = chunks[0].page_content[:900].strip()

# Console output
print("=" * 60)
print(" ✂️ DOCUMENT CHUNKING REPORT")
print("=" * 60)
print(f" ► Total Chunks Generated : {len(chunks)}")
print(f" ► Chunk Size Config      : 1000 chars (Overlap: 150)")
print(f" ► Start Index Preserved  : {chunks[0].metadata.get('start_index', 'N/A')}")
print("-" * 60)
print(" 📝 FIRST CHUNK PREVIEW (First 900 Characters)")
print("-" * 60)
print(chunk_preview)
print("=" * 60)

 ✂️ DOCUMENT CHUNKING REPORT
 ► Total Chunks Generated : 28
 ► Chunk Size Config      : 1000 chars (Overlap: 150)
 ► Start Index Preserved  : 0
------------------------------------------------------------
 📝 FIRST CHUNK PREVIEW (First 900 Characters)
------------------------------------------------------------
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCust

## Local Embeddings

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings.embed_query("Who is Messi??")
print("Embedding dimensions:", len(sample_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4636.88it/s]


Embedding dimensions: 384


## Pinecone Vector DB

In [4]:
import os
import time
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

print("=" * 60)
print(" 🌲 PINECONE VECTOR DATABASE SETUP")
print("=" * 60)

# Create the index only if it does not already exist.
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f" ► Creating new index '{INDEX_NAME}'...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,  # all-MiniLM-L6-v2 embedding dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # Wait until Pinecone reports the new index as ready.
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        print("   ⏳ Waiting for index initialization...")
        time.sleep(1)

print(f" ► Index Status           : Ready ('{INDEX_NAME}')")
print(" ► Uploading Chunks       : Processing vector embeddings...")

# Upload the document chunks and create the LangChain vector store.
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("-" * 60)
print(" ✅ CONFIGURATION & INGESTION COMPLETE")
print("-" * 60)
print(f" ► Vector Store Namespace : {NAMESPACE}")
print(" ► Top-K Retrieval Depth  : 4")
print(" ► Pipeline Status        : Retriever ready for queries")
print("=" * 60)

 🌲 PINECONE VECTOR DATABASE SETUP
 ► Creating new index 'agentic-rag-kb'...
 ► Index Status           : Ready ('agentic-rag-kb')
 ► Uploading Chunks       : Processing vector embeddings...
------------------------------------------------------------
 ✅ CONFIGURATION & INGESTION COMPLETE
------------------------------------------------------------
 ► Vector Store Namespace : langgraph-agentic-rag
 ► Top-K Retrieval Depth  : 4
 ► Pipeline Status        : Retriever ready for queries


In [5]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

INDEX_NAME = "agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

print("=" * 60)
print(" 🔄 LOADING EXISTING PINECONE INDEX")
print("=" * 60)

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Load existing Pinecone index
index = pc.Index(INDEX_NAME)

# Connect existing index with LangChain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print(f" ► Index Loaded           : {INDEX_NAME}")
print(f" ► Namespace             : {NAMESPACE}")
print("-" * 60)
print(" ✅ RETRIEVAL PIPELINE READY")
print("-" * 60)
print(" ► Top-K Retrieval Depth  : 4")
print(" ► Pipeline Status        : Connected & Ready for queries")
print("=" * 60)

 🔄 LOADING EXISTING PINECONE INDEX
 ► Index Loaded           : agentic-rag-kb
 ► Namespace             : langgraph-agentic-rag
------------------------------------------------------------
 ✅ RETRIEVAL PIPELINE READY
------------------------------------------------------------
 ► Top-K Retrieval Depth  : 4
 ► Pipeline Status        : Connected & Ready for queries


## Test KB Retrieval

In [6]:
test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, doc in enumerate(kb_docs, 1):
    print(f"\n--- KB RESULT {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:800])


--- KB RESULT 1 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageConceptsSetupSet up LangSmithPreprocess documentsCreate a retriever toolGenerate a query or respondGrade documentsRewrite the questionGenerate an answerAssemble the graphRun the agentic RAGSee alsoTutorialsLangGraphBuild a custom RAG agent with LangGraphCopy pageCopy pageBuild a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.Copy pageCopy pageBuild a retrieval agent with LangGraph that decides when to search a vector store versus answering the user directly.

--- KB RESULT 2 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is c

## Groq LLM

In [7]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

print(llm.invoke("Explain birds in one sentence.").content)

Birds are warm‑blooded, feathered vertebrates that lay eggs, possess wings, and are capable of flight—though some species have evolved to be flightless.


## Tavily

In [8]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=4,
    topic="general",
    include_answer=True,
    include_raw_content=False,
)

print("Tavily search tool ready.")

Tavily search tool ready.


## Structured Decisions

In [9]:
from typing import List, Literal, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.documents import Document


class RouteDecision(BaseModel):
    route: Literal["kb", "direct"] = Field(
        description="Use kb for questions needing Agentic RAG docs; direct for greetings/simple chat."
    )


class EvidenceGrade(BaseModel):
    grade: Literal["good", "weak"] = Field(
        description="good means evidence can answer the question; weak means not enough evidence."
    )


class AgentState(TypedDict):
    question: str
    current_query: str
    kb_docs: List[Document]
    web_results: str
    kb_grade: str
    web_grade: str
    answer: str
    source_used: str
    retry_count: int

## 1-Route the Question

In [10]:
router_llm = llm.with_structured_output(RouteDecision, method="json_mode")

def route_question(state: AgentState):
    question = state["question"]

    decision = router_llm.invoke(f'''
You are a router for an Agentic RAG assistant.

Route to "kb" if the user asks about:
- Agentic RAG
- LangGraph Agentic RAG workflow
- retrieval grading
- query rewriting
- RAG architecture
- retriever tools
- web fallback in RAG

Route to "direct" only for greetings, thanks, or very simple conversation.

Question:
{question}

Return your response as valid JSON.
Example:
{{"route": "kb"}}
''')

    print("[Router]", decision.route)

    return {
        "current_query": question,
        "source_used": decision.route,
    }


def route_after_router(state: AgentState) -> Literal["retrieve_kb", "direct_answer"]:
    if state["source_used"] == "kb":
        return "retrieve_kb"
    return "direct_answer"

## 2-Retrieve from the Private KB

In [11]:
def retrieve_kb(state: AgentState):
    query = state["current_query"]
    docs = retriever.invoke(query)

    print(f"[KB Retriever] Query: {query}")
    print(f"[KB Retriever] Retrieved: {len(docs)} chunks")

    return {"kb_docs": docs}